In [0]:
%sql
USE financial_transactions.silver

## 1. Volume e unicidade
Verifica quantos registros existem e quantos ids são únicos. Se esses números forem diferentes, há ids repetidos.

**Ideal:** linhas e ids iguais em todas as bases, ou seja, nenhum registro repetido.

In [0]:
%sql
WITH contagem AS (
  SELECT 'users'        AS base, COUNT(*) AS linhas, COUNT(DISTINCT id)             AS ids_unicos FROM users
  UNION ALL
  SELECT 'cards'        AS base, COUNT(*) AS linhas, COUNT(DISTINCT id)             AS ids_unicos FROM cards
  UNION ALL
  SELECT 'transactions' AS base, COUNT(*) AS linhas, COUNT(DISTINCT id)             AS ids_unicos FROM transactions
  UNION ALL
  SELECT 'mcc_codes'    AS base, COUNT(*) AS linhas, COUNT(DISTINCT mcc)            AS ids_unicos FROM mcc_codes
  UNION ALL
  SELECT 'fraud_labels' AS base, COUNT(*) AS linhas, COUNT(DISTINCT transaction_id) AS ids_unicos FROM fraud_labels
)
SELECT
  base,
  linhas,
  ids_unicos,
  CASE WHEN linhas = ids_unicos THEN 'igual'      ELSE 'diferente' END AS resultado,
  CASE WHEN linhas = ids_unicos THEN 'prosseguir' ELSE 'avaliar'   END AS acao
FROM contagem

base,linhas,ids_unicos,resultado,acao
users,2000,2000,igual,prosseguir
cards,6146,6146,igual,prosseguir
transactions,13295276,13295276,igual,prosseguir
mcc_codes,109,109,igual,prosseguir
fraud_labels,8907770,8907770,igual,prosseguir


## 2. Completude
Conta quantos campos estão vazios em cada coluna.

**Ideal:** 0 vazios, com exceção dos explicados nas transações: estado, país e CEP vazios nas compras online e no exterior.

In [0]:
from pyspark.sql import functions as F

tabelas = ["users", "cards", "transactions", "mcc_codes", "fraud_labels"]
resumo = []

for tabela in tabelas:
    df = spark.table(tabela)
    colunas = [c for c in df.columns if not c.startswith("_")]  # ignora as colunas de controle

    # Conta os campos nulos ou vazios de cada coluna, lendo a tabela uma vez só
    vazios = df.select([
        F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
        for c in colunas
    ]).first().asDict()

    colunas_com_vazio = [f"{c} ({vazios[c]})" for c in colunas if vazios[c] > 0]
    total_vazios = sum(vazios.values())

    resumo.append((
        tabela,
        df.count(),
        total_vazios,
        ", ".join(colunas_com_vazio) or "-",
        "completo" if total_vazios == 0 else "incompleto",
        "prosseguir" if total_vazios == 0 else "avaliar",
    ))

display(spark.createDataFrame(resumo, ["base", "linhas", "campos_vazios", "colunas_com_vazio", "resultado", "acao"]))

base,linhas,campos_vazios,colunas_com_vazio,resultado,acao
users,2000,0,-,completo,prosseguir
cards,6146,0,-,completo,prosseguir
transactions,13295276,4869088,"merchant_state (1652696), merchant_country (1563696), zip (1652696)",incompleto,avaliar
mcc_codes,109,0,-,completo,prosseguir
fraud_labels,8907770,0,-,completo,prosseguir


## 3. Formato
Verifica se os valores preenchidos estão no formato esperado (número, dinheiro ou data) em todas as bases.

**Ideal:** 0 colunas com tipo errado e 0 falhas de conversão em todas as bases.

In [0]:
from pyspark.sql import functions as F

# Tipo esperado de cada coluna da Silver e a coluna de origem na Bronze
# (None = coluna nova ou com vazios já explicados, que não entra na comparação com a Bronze)
tipos = {
    "users": {
        "id": ("int", "id"), "current_age": ("int", "current_age"), "retirement_age": ("int", "retirement_age"),
        "birth_year": ("int", "birth_year"), "birth_month": ("int", "birth_month"), "gender": ("string", "gender"),
        "city": ("string", None), "state": ("string", None),
        "per_capita_income": ("decimal(12,2)", "per_capita_income"), "yearly_income": ("decimal(12,2)", "yearly_income"),
        "total_debt": ("decimal(12,2)", "total_debt"), "credit_score": ("int", "credit_score"),
        "num_credit_cards": ("int", "num_credit_cards"),
    },
    "cards": {
        "id": ("int", "id"), "client_id": ("int", "client_id"), "card_number_masked": ("string", "card_number"),
        "expires_date": ("date", "expires"), "has_chip": ("boolean", "has_chip"),
        "num_cards_issued": ("int", "num_cards_issued"), "credit_limit": ("decimal(12,2)", "credit_limit"),
        "acct_open_date": ("date", "acct_open_date"), "year_pin_last_changed": ("int", "year_pin_last_changed"),
        "card_on_dark_web": ("boolean", "card_on_dark_web"),
    },
    "transactions": {
        "id": ("bigint", "id"), "transaction_date": ("timestamp", "date"), "client_id": ("int", "client_id"),
        "card_id": ("int", "card_id"), "amount": ("decimal(12,2)", "amount"), "merchant_id": ("int", "merchant_id"),
        "zip": ("string", "zip"), "mcc": ("int", "mcc"),
        "channel": ("string", None), "merchant_country": ("string", None),
        "errors": ("string", None), "alert_type_mismatch": ("boolean", None),
    },
    "mcc_codes": {"mcc": ("int", "mcc"), "description": ("string", "descricao")},
    "fraud_labels": {"transaction_id": ("bigint", "transaction_id"), "is_fraud": ("boolean", "is_fraud")},
}

# Colunas em que vazio é esperado (já explicado na completude)
vazio_esperado = {"merchant_country"}

resumo = []

for tabela, colunas in tipos.items():
    silver = spark.table(f"financial_transactions.silver.{tabela}")
    bronze = spark.table(f"financial_transactions.bronze.{tabela}")

    # 1. Tipo: compara o tipo de cada coluna da Silver com o esperado
    tipo_atual = {campo.name: campo.dataType.simpleString() for campo in silver.schema}
    tipo_errado = [f"{c} ({tipo_atual.get(c, 'não existe')})" for c, (tipo, _) in colunas.items() if tipo_atual.get(c) != tipo]

    # Só segue com as colunas que existem na Silver
    existentes = {c: origem for c, (_, origem) in colunas.items() if c in tipo_atual}

    # 2. Conversão: vazios na Silver menos vazios na coluna de origem da Bronze
    vazios_silver = silver.select([
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in existentes
    ]).first().asDict()

    vazios_bronze = bronze.select([
        F.sum(F.when(F.col(origem).isNull() | (F.trim(F.col(origem)) == ""), 1).otherwise(0)).alias(c)
        for c, origem in existentes.items() if origem
    ]).first().asDict()

    falhas = {
        c: 0 if c in vazio_esperado else vazios_silver[c] - vazios_bronze.get(c, 0)
        for c in existentes
    }
    colunas_com_falha = [f"{c} ({falhas[c]})" for c in existentes if falhas[c] > 0]
    total_falhas = sum(v for v in falhas.values() if v > 0)

    ok = not tipo_errado and total_falhas == 0
    resumo.append((
        tabela,
        len(colunas),
        ", ".join(tipo_errado) or "-",
        total_falhas,
        ", ".join(colunas_com_falha) or "-",
        "válido" if ok else "inválido",
        "prosseguir" if ok else "avaliar",
    ))

display(spark.createDataFrame(resumo, ["base", "colunas_verificadas", "tipo_errado", "falhas_conversao", "colunas_com_falha", "resultado", "acao"]))

base,colunas_verificadas,tipo_errado,falhas_conversao,colunas_com_falha,resultado,acao
users,13,-,0,-,válido,prosseguir
cards,10,-,0,-,válido,prosseguir
transactions,12,-,0,-,válido,prosseguir
mcc_codes,2,-,0,-,válido,prosseguir
fraud_labels,2,-,0,-,válido,prosseguir


## 4. Valores possíveis
Verifica se os valores estão dentro de faixas reais (idade, score, limite, renda) e se os textos têm apenas as categorias esperadas.

**Ideal:** 0 valores fora do esperado. Quando houver, cada caso é avaliado: corrigir, aceitar com alerta ou enviar para a quarentena.

In [0]:
from pyspark.sql import functions as F

# Regras de valores possíveis na Silver: cada regra descreve como um valor CORRETO deve ser
regras = {
    "users": {
        "idade entre 18 e 100":        "current_age BETWEEN 18 AND 100",
        "aposentadoria entre 50 e 80": "retirement_age BETWEEN 50 AND 80",
        "mês entre 1 e 12":            "birth_month BETWEEN 1 AND 12",
        "score entre 300 e 850":       "credit_score BETWEEN 300 AND 850",
        "ao menos 1 cartão":           "num_credit_cards >= 1",
        "renda per capita > 0":        "per_capita_income > 0",
        "renda anual > 0":             "yearly_income > 0",
        "dívida >= 0":                 "total_debt >= 0",
        "gênero Female ou Male":       "gender IN ('Female', 'Male')",
        "cidade preenchida":           "city IS NOT NULL",
        "estado preenchido":           "state IS NOT NULL",
    },
    "cards": {
        "limite > 0":                  "credit_limit > 0",
        "bandeira conhecida":          "card_brand IN ('Visa', 'Mastercard', 'Amex', 'Discover')",
        "tipo conhecido":              "card_type IN ('Debit', 'Credit', 'Debit (Prepaid)')",
        "chip verdadeiro ou falso":    "has_chip IS NOT NULL",
        "dark web verdadeiro ou falso": "card_on_dark_web IS NOT NULL",
        "ao menos 1 cartão emitido":   "num_cards_issued >= 1",
    },
    "transactions": {
        "tipo de transação conhecido": "transaction_type IN ('Swipe Transaction', 'Chip Transaction', 'Online Transaction')",
        "valor diferente de zero":     "amount <> 0",
        "canal conhecido":             "channel IN ('Online', 'Presencial EUA', 'Presencial exterior')",
        "tipo coerente com o canal":   "NOT alert_type_mismatch",
    },
    "fraud_labels": {
        "rótulo verdadeiro ou falso":  "is_fraud IS NOT NULL",
    },
}

resumo = []

for tabela, regras_tabela in regras.items():
    df = spark.table(f"financial_transactions.silver.{tabela}")

    # Conta, para cada regra, quantos valores NÃO atendem à regra
    # coalesce(..., False): se o valor estiver vazio, a regra conta como não atendida
    fora = df.select([
        F.sum(F.when(~F.coalesce(F.expr(condicao), F.lit(False)), 1).otherwise(0)).alias(nome)
        for nome, condicao in regras_tabela.items()
    ]).first().asDict()

    regras_com_problema = [f"{nome} ({fora[nome]})" for nome in regras_tabela if fora[nome] > 0]
    total_fora = sum(fora.values())

    resumo.append((
        tabela,
        len(regras_tabela),
        total_fora,
        ", ".join(regras_com_problema) or "-",
        "dentro do esperado" if total_fora == 0 else "fora do esperado",
        "prosseguir" if total_fora == 0 else "avaliar",
    ))

display(spark.createDataFrame(resumo, ["base", "regras_verificadas", "valores_fora", "regras_com_problema", "resultado", "acao"]))

base,regras_verificadas,valores_fora,regras_com_problema,resultado,acao
users,11,16,"idade entre 18 e 100 (1), renda per capita > 0 (15)",fora do esperado,avaliar
cards,6,31,limite > 0 (31),fora do esperado,avaliar
transactions,4,5788,tipo coerente com o canal (5788),fora do esperado,avaliar
fraud_labels,1,0,-,dentro do esperado,prosseguir


## 5. Coerência entre colunas
Verifica se informações relacionadas fazem sentido juntas, como idade e ano de nascimento, renda per capita e renda anual, e abertura e vencimento do cartão.

**Ideal:** 0 registros incoerentes. Regras marcadas como alerta indicam situações possíveis, que são acompanhadas, mas não bloqueiam o registro.

In [0]:
from pyspark.sql import functions as F

# Regras de coerência na Silver: cada regra cruza duas ou mais colunas e descreve como um registro CORRETO deve ser
regras = {
    "users": {
        "idade coerente com o nascimento":
            "current_age = 2020 - birth_year - CASE WHEN birth_month >= 3 THEN 1 ELSE 0 END",
        "renda per capita <= renda anual":
            "per_capita_income <= yearly_income",
        "idade <= aposentadoria (alerta)":
            "current_age <= retirement_age",
    },
    "cards": {
        "abertura antes do mês de vencimento":
            "acct_open_date < TRUNC(expires_date, 'MM')",
        "PIN alterado após a abertura":
            "year_pin_last_changed >= YEAR(acct_open_date)",
    },
    "transactions": {
        "data até o fim de 2019":
            "transaction_date < '2020-01-01'",
    },
}

resumo = []

for tabela, regras_tabela in regras.items():
    df = spark.table(f"financial_transactions.silver.{tabela}")

    # Conta, para cada regra, quantos registros NÃO atendem à regra
    # coalesce(..., False): se algum valor estiver vazio, a regra conta como não atendida
    fora = df.select([
        F.sum(F.when(~F.coalesce(F.expr(condicao), F.lit(False)), 1).otherwise(0)).alias(nome)
        for nome, condicao in regras_tabela.items()
    ]).first().asDict()

    regras_com_problema = [f"{nome} ({fora[nome]})" for nome in regras_tabela if fora[nome] > 0]
    total_fora = sum(fora.values())

    resumo.append((
        tabela,
        len(regras_tabela),
        total_fora,
        ", ".join(regras_com_problema) or "-",
        "coerente" if total_fora == 0 else "incoerente",
        "prosseguir" if total_fora == 0 else "avaliar",
    ))

display(spark.createDataFrame(resumo, ["base", "regras_verificadas", "registros_incoerentes", "regras_com_problema", "resultado", "acao"]))

base,regras_verificadas,registros_incoerentes,regras_com_problema,resultado,acao
users,3,311,"renda per capita <= renda anual (34), idade <= aposentadoria (alerta) (277)",incoerente,avaliar
cards,2,248,abertura antes do mês de vencimento (248),incoerente,avaliar
transactions,1,0,-,coerente,prosseguir


## 6. Integridade entre tabelas
Verifica se as tabelas se conectam corretamente: se todo cartão tem cliente, se toda transação aponta para cliente, cartão e categoria existentes, e se os rótulos de fraude correspondem a transações reais.

**Ideal:** 0 registros sem correspondência em todas as verificações.


In [0]:
%sql
WITH checagens AS (
  -- Todo cartão pertence a um cliente cadastrado?
  SELECT 'cards' AS base, 'cliente existe em users' AS verificacao,
         COUNT_IF(u.id IS NULL) AS sem_correspondencia
  FROM cards c
  LEFT JOIN users u ON c.client_id = u.id

  UNION ALL
  -- A quantidade de cartões do cliente bate com a tabela de cartões?
  SELECT 'users', 'quantidade de cartões igual a cards',
         COUNT_IF(q.qtd IS NULL OR INT(u.num_credit_cards) <> q.qtd)
  FROM users u
  LEFT JOIN (SELECT client_id, COUNT(*) AS qtd FROM cards GROUP BY client_id) q
    ON u.id = q.client_id

  UNION ALL
  -- Toda transação é de um cliente cadastrado?
  SELECT 'transactions', 'cliente existe em users',
         COUNT_IF(u.id IS NULL)
  FROM transactions t
  LEFT JOIN users u ON t.client_id = u.id

  UNION ALL
  -- Toda transação usa um cartão cadastrado?
  SELECT 'transactions', 'cartão existe em cards',
         COUNT_IF(c.id IS NULL)
  FROM transactions t
  LEFT JOIN cards c ON t.card_id = c.id

  UNION ALL
  -- O cartão usado pertence ao mesmo cliente da transação?
  SELECT 'transactions', 'cartão pertence ao cliente',
         COUNT_IF(c.client_id <> t.client_id)
  FROM transactions t
  JOIN cards c ON t.card_id = c.id

  UNION ALL
  -- Toda transação tem uma categoria de comércio cadastrada?
  SELECT 'transactions', 'mcc existe em mcc_codes',
         COUNT_IF(m.mcc IS NULL)
  FROM transactions t
  LEFT JOIN mcc_codes m ON t.mcc = m.mcc

  UNION ALL
  -- Todo rótulo de fraude aponta para uma transação existente?
  SELECT 'fraud_labels', 'transação existe em transactions',
         COUNT_IF(t.id IS NULL)
  FROM fraud_labels f
  LEFT JOIN transactions t ON f.transaction_id = t.id
)
SELECT
  base,
  verificacao,
  sem_correspondencia,
  CASE WHEN sem_correspondencia = 0 THEN 'íntegro'    ELSE 'com falhas' END AS resultado,
  CASE WHEN sem_correspondencia = 0 THEN 'prosseguir' ELSE 'avaliar'    END AS acao
FROM checagens

base,verificacao,sem_correspondencia,resultado,acao
cards,cliente existe em users,0,íntegro,prosseguir
users,quantidade de cartões igual a cards,0,íntegro,prosseguir
transactions,cliente existe em users,0,íntegro,prosseguir
transactions,cartão existe em cards,0,íntegro,prosseguir
transactions,cartão pertence ao cliente,0,íntegro,prosseguir
transactions,mcc existe em mcc_codes,0,íntegro,prosseguir
fraud_labels,transação existe em transactions,0,íntegro,prosseguir
